In [1]:
from skimage import feature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB


In [2]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [3]:
root_folder='../data/test_data/embedding/birdnet/'

In [4]:
df_pos = get_embedding_birdnet(root_folder, 1)
df_neg = get_embedding_birdnet(root_folder, 0)

df_pos['target'] = 1
df_neg['target'] = 0

In [5]:
df_test = pd.concat([df_pos, df_neg], ignore_index=True)

Perform 5-fold inference

SVM

In [6]:
import os
import joblib
import numpy as np

from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix

# Prepare test data
X_test = np.vstack(df_test["embedding"].values)
y_test = df_test["target"].values
dataset = "ds1"

accuracies = []
aps = []

for fold in range(1, 6):

    model = joblib.load(f"../notebooks/saved_models/{dataset}_svm_fold_{fold}.joblib")

    y_pred = model.predict(X_test)
    y_score = model.decision_function(X_test)

    acc = accuracy_score(y_test, y_pred)
    ap = average_precision_score(y_test, y_score)
    cm = confusion_matrix(y_test, y_pred)

    accuracies.append(acc)
    aps.append(ap)

    print(f"\n===== Fold {fold} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"AP       : {ap:.4f}")
    print(cm)

print("\n==========================")
print("Accuracy per model:", np.round(accuracies, 4))
print(f"Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies, ddof=1):.4f}")

print("AP per model:", np.round(aps, 4))
print(f"AP: {np.mean(aps):.4f} ± {np.std(aps, ddof=1):.4f}")



===== Fold 1 =====
Accuracy : 0.8865
AP       : 0.9792
[[ 34  21]
 [  5 169]]

===== Fold 2 =====
Accuracy : 0.8821
AP       : 0.9761
[[ 35  20]
 [  7 167]]

===== Fold 3 =====
Accuracy : 0.8734
AP       : 0.9772
[[ 35  20]
 [  9 165]]

===== Fold 4 =====
Accuracy : 0.8734
AP       : 0.9716
[[ 32  23]
 [  6 168]]

===== Fold 5 =====
Accuracy : 0.8734
AP       : 0.9766
[[ 35  20]
 [  9 165]]

Accuracy per model: [0.8865 0.8821 0.8734 0.8734 0.8734]
Accuracy: 0.8777 ± 0.0062
AP per model: [0.9792 0.9761 0.9772 0.9716 0.9766]
AP: 0.9761 ± 0.0028


Random Forest

XGBoost